In [1]:
!pip install stable-baselines3[extra]

In [2]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
import gymnasium as gym
from gym import Env
from gym.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete
import numpy as np
import random
import os

Types of spaces

In [4]:
Discrete(3).sample()

1

In [5]:
Discrete(2).sample()

1

In [6]:
Box(0,1,shape=(3,)).sample()

array([0.4815985 , 0.13307592, 0.3173354 ], dtype=float32)

Box(0,1) = Uniform distribution between 0 and 1

shape=(3,) = Generate 3 independent samples

.sample() = Draw the random numbers

Wrappers helps in combining spaces

In [7]:
#wrapper spaces
Tuple((Discrete(3),Box(0,1,shape=(3,)))).sample()  # 1st component: discrete action/observation {0,1,2};  # 2nd component: 3 continuous values [0,1]

(1, array([0.02623475, 0.16602758, 0.5029726 ], dtype=float32))

In [8]:
#wrapper space
Dict({
    "position": Discrete(10),      # int: 0-9
    "velocity": Box(-1,1,shape=(2,))  # array: [-1,-1] to [1,1]
})

#Observation = dictionary with named fields (position as integer, velocity as 2D vector)


Dict('position': Discrete(10), 'velocity': Box(-1.0, 1.0, (2,), float32))

In [9]:
MultiBinary(4)


MultiBinary(4)

MultiDiscrete([5, 3, 3])

       ↑   ↑  ↑
       
     3 vars: 0-4, 0-2, 0-2


In [10]:
MultiDiscrete([5,3,3]).sample()

array([0, 0, 1])

Building an environment!

In [11]:
#building an agent that gives us the best shower possible
#random temperatures
#37 to 39 degrees

In [12]:
Discrete(3).sample()

0

0 ---> on

1 ----> off

2 ----> leave it unchanged

In [13]:
Box(0,1,shape=(3,)).sample()

array([0.21018936, 0.89543736, 0.53916276], dtype=float32)

In [53]:
import gymnasium as gym
from gymnasium import spaces  # ← CRITICAL: gymnasium spaces

class shower_env(gym.Env):
  def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(3)  # ← gymnasium.spaces.Discrete
        # Changed observation space to Box for continuous temperature that can go out of bounds
        self.observation_space = spaces.Box(low=np.array([-50.0]), high=np.array([100.0]), shape=(1,), dtype=np.float32)
        self.state = np.array([37.0]).astype(np.float32)  # starting temp as float array
        self.shower_length = 60
  def step(self, action):
    self.state += action - 1      # 0→-1, 1→0, 2→+1
    self.shower_length -= 1

    if 37 <= self.state <= 39:
        reward = 1
    else:
        reward = -1

    done = self.shower_length <= 0
    info = {}
    return self.state, reward, done, False, info  # gymnasium 5-return format

  def render(self):
      #implement viz
    pass # Added pass as a placeholder for render method
  def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    self.state = np.array([38.0], dtype=np.float32)  # inside [37,39]
    self.shower_length = 60
    info = {}
    return self.state, info


In [54]:
env=shower_env()

print()
env.action_space.sample()

np.int64(2)

In [55]:
env.observation_space.sample()

array([36.55813], dtype=float32)

In [56]:
#testing
episodes=8
for episode in range(1,episodes+1):
  obs, info = env.reset()
  terminated=False
  truncated=False
  score=0
  while not terminated and not truncated:
    action=env.action_space.sample()
    obs,reward,terminated,truncated,info=env.step(action)
    score+=reward
  print(episode,':',score)

1 : -50
2 : -4
3 : -16
4 : -20
5 : -6
6 : -26
7 : -18
8 : -18


In [18]:
!pip install gymnasium[box2d]  # uninstall gym first if needed


In [58]:
#training model
from stable_baselines3.common.vec_env import DummyVecEnv
log_path=os.path.join('training','logs')
from stable_baselines3 import A2C

model = A2C('MlpPolicy', env,
            policy_kwargs={'net_arch': [64, 64]},  # bigger net
            ent_coef=0.01)  # ← add entropy bonus
model.learn(total_timesteps=10000)

In [59]:
evaluate_policy(model, env, n_eval_episodes=10, render=True)

(np.float64(60.0), np.float64(0.0))

In [60]:
env = shower_env()
obs, info = env.reset()
total_reward = 0
for t in range(60):
    action = 1   # keep temp
    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f"t={t}, state={env.state}, reward={reward}")
    if done or truncated:
        break

print("Total =", total_reward)


t=0, state=[38.], reward=1
t=1, state=[38.], reward=1
t=2, state=[38.], reward=1
t=3, state=[38.], reward=1
t=4, state=[38.], reward=1
t=5, state=[38.], reward=1
t=6, state=[38.], reward=1
t=7, state=[38.], reward=1
t=8, state=[38.], reward=1
t=9, state=[38.], reward=1
t=10, state=[38.], reward=1
t=11, state=[38.], reward=1
t=12, state=[38.], reward=1
t=13, state=[38.], reward=1
t=14, state=[38.], reward=1
t=15, state=[38.], reward=1
t=16, state=[38.], reward=1
t=17, state=[38.], reward=1
t=18, state=[38.], reward=1
t=19, state=[38.], reward=1
t=20, state=[38.], reward=1
t=21, state=[38.], reward=1
t=22, state=[38.], reward=1
t=23, state=[38.], reward=1
t=24, state=[38.], reward=1
t=25, state=[38.], reward=1
t=26, state=[38.], reward=1
t=27, state=[38.], reward=1
t=28, state=[38.], reward=1
t=29, state=[38.], reward=1
t=30, state=[38.], reward=1
t=31, state=[38.], reward=1
t=32, state=[38.], reward=1
t=33, state=[38.], reward=1
t=34, state=[38.], reward=1
t=35, state=[38.], reward=1
t=

For a richer notion of “learning” you would need a harder env (noise, drifting temperature, random starts, etc.), but for this simple tutorial env, constant state at 38 with reward 1 means the agent has converged to the optimal policy.

